In [5]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
from tqdm import tqdm

In [10]:
from netgen.stl import STLGeometry
from ngsolve import Mesh as NGSolveMesh

# The following code took hours of trial and error to get right
# DO NOT GET INTO TRYING TO CONVERT BETWEEN DIFFERENT MESH FORMATS!!!!!
# Time spent on this is time wasted:
# Wasted man hours 12.5
geo = STLGeometry("Bunny.stl") 
ngmesh = geo.GenerateMesh(maxh=8)
for el in ngmesh.Elements2D():
    for v in el.vertices:
        x, y, z = ngmesh[v]
        if z < 8:
            el.index = 1
            break
        if z > 109:
            el.index = 2
            break
        else:
            el.index = 3
            
ngmesh.SetBCName(0, "bottom")
ngmesh.SetBCName(2, "middle")
ngmesh.SetBCName(1, "top")

# Schritt 3: Mesh in NGSolve importieren
mesh = NGSolveMesh(ngmesh)
mesh.GetBoundaries()

('bottom', 'middle', 'top')

In [11]:
festest = VectorH1(mesh, order=3)
gftest = GridFunction(festest, name="test")

In [12]:
gftest.Set(CF((1,1,1)), definedon=mesh.Boundaries("bottom"))
Draw(gftest, mesh, height="2vh")

WebGuiWidget(layout=Layout(height='2vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

BaseWebGuiScene

In [13]:
gftest.Set(CF((1,1,1)), definedon=mesh.Boundaries("middle"))
Draw(gftest, mesh, height="2vh")

WebGuiWidget(layout=Layout(height='2vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

BaseWebGuiScene

In [ ]:
gftest.Set(CF((1,1,1)), definedon=mesh.Boundaries("top"))
Draw(gftest, mesh, height="2vh")

In [5]:
dt = 0.2
beta = .25 # 1/4 limit for unconditional stability, set higher for more damping
gamma = .5 # trapezoidal rule

eta = 1e-2 # viscoealstic damping CAREFUL: CFL is very strict here or something else limiting (maybe mesh size???). i dont understand but cannot go above 1% for any fucking reason. else crashes ...

E, nu = 210, 0.2
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

def Stress(strain):
    return 2*mu*strain + lam*Trace(strain)*Id(3)

In [6]:
t = Parameter(0.0)
fes = VectorH1(mesh, order=3, dirichlet="fix")
u,v = fes.TnT()
gfu = GridFunction(fes)
gfv = GridFunction(fes)
gfa = GridFunction(fes)

with TaskManager():
    m_int = u*v*dx
    m = BilinearForm(m_int)
    a_int = InnerProduct(Stress(Sym(Grad(u))), Sym(Grad(v))).Compile()*dx
    a = BilinearForm(a_int)
    c_int = 2 * InnerProduct( eta * Sym(Grad(u)), Sym(Grad(v))).Compile()*dx
    c = BilinearForm(c_int)
    mstar = BilinearForm(fes)
    mstar += m_int
    mstar += - gamma * dt * c_int
    mstar += beta * dt**2 * a_int
    pre = MultiGridPreconditioner(mstar, "bddc") # Adds factor 8 Speedup for some reason ...
    mstar.Assemble()
    

TypeError: __init__(): incompatible constructor arguments. The following argument types are supported:
    1. ngsolve.comp.VectorH1(mesh: ngsolve.comp.Mesh, **kwargs)

Invoked with: <netgen.libngpy._meshing.Mesh object at 0x7f5866e293b0>; kwargs: order=3, dirichlet='fix'

In [22]:
omega = 0.4*3.5 # resoncance with  is around 1.5
force = CF( (0,0,5))*cos(2*np.pi*omega*t) 
# force = CF( (2,0,0))
f = LinearForm(force*v*ds("force"))

In [23]:
from ngsolve.krylovspace import CGSolver

acc_inv = CGSolver(mstar.mat, pre, printrates=False, tol=1e-8)

In [24]:
gfa.Set(CF((0,0,0)))
gfv.Set(CF((0,0,0)))
gfu.Set(CF((0,0,0)))

In [25]:
gfut = GridFunction(gfu.space,multidim=0)

fesstress = MatrixValued(H1(mesh,order=3), symmetric=True)
gfstress = GridFunction(fesstress)
gfstresst = GridFunction(fesstress, multidim=0)

tend = t.Get()+4
steps = int((tend - t.Get()) / dt)

with TaskManager():
    for i in tqdm(range(steps), desc="Time-stepping", ncols=100, unit="step"):
        m.Assemble()
        c.Assemble()
        a.Assemble()
        f.Assemble()

        rhs_acceleration = f.vec - a.mat*(gfu.vec + dt*gfv.vec + .5*dt**2*(1-2*beta)*gfa.vec) + c.mat*(gfv.vec + (1-gamma)*gfa.vec) 
        
        new_acceleration = acc_inv * rhs_acceleration
        
        gfu.vec.data = gfu.vec + dt*gfv.vec + 0.5*dt**2*((1-2*beta)*gfa.vec + 2*beta*new_acceleration)
        
        gfv.vec.data = gfv.vec + dt*((1-gamma)*gfa.vec + gamma*new_acceleration)
        
        gfa.vec.data = new_acceleration
        
        if i%1 == 0:
            gfut.AddMultiDimComponent(gfu.vec)
            
            gfstress.Interpolate(Stress(Sym(Grad(gfu))))
            
            gfstresst.AddMultiDimComponent(gfstress.vec)
        
        t.Set(t.Get()+dt)
    

Time-stepping: 100%|██████████████████████████████████████████████| 20/20 [00:37<00:00,  1.87s/step]


In [26]:
with TaskManager():
    fesstress = MatrixValued(H1(mesh,order=3), symmetric=True)
    gfstress = GridFunction(fesstress)
    gfstress.Interpolate (Stress(Sym(Grad(gfu))))

In [27]:
Draw (gfut, mesh, height="3vh", deformation=gfut, interpolate_multidim=True, animate=True)

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

BaseWebGuiScene

In [28]:
Draw(gfstresst, mesh, height="3vh", deformation=gfut, interpolate_multidim=True, animate=True)

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

BaseWebGuiScene